In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from pathlib import Path

# Project root (go up one level from notebooks/)
ROOT = Path("..").resolve()
RAW  = ROOT / "data" / "raw"

TRAIN_IMG = RAW / "WIDER_train" / "images"
VAL_IMG   = RAW / "WIDER_val"   / "images"
ANNO_DIR  = RAW / "wider_face_split"

print("ROOT:", ROOT)
print("Train images exist:", TRAIN_IMG.exists())
print("Val images exist:  ", VAL_IMG.exists())
print("Annotations exist: ", ANNO_DIR.exists())

In [ ]:
face_data  = train_anno["face_bbx_list"]
event_list = train_anno["event_list"]
file_list  = train_anno["file_list"]

print(f"Total event categories: {len(event_list)}")
print(f"Sample events: {event_list[:5]}")

In [ ]:
# See exactly what's inside your .mat file
print("All keys in .mat file:")
for key in train_anno.keys():
    print(f"  '{key}'")

In [ ]:
total_images = 0
total_faces  = 0

for i in range(len(event_list)):
    files = file_list[i]
    boxes = face_data[i]
    for j in range(len(files)):
        total_images += 1
        total_faces  += len(boxes[j])

print(f"Total training images : {total_images:,}")
print(f"Total face annotations: {total_faces:,}")
print(f"Avg faces per image   : {total_faces/total_images:.1f}")

In [ ]:
def show_samples(n_events=3, n_images=2):
    fig, axes = plt.subplots(n_events, n_images, figsize=(14, 10))
    fig.suptitle("WIDER FACE — Sample Annotations", fontsize=14)

    for i, event in enumerate(event_list[:n_events]):
        files = file_list[i]

        for j in range(min(n_images, len(files))):
            img_path = TRAIN_IMG / event / (files[j] + ".jpg")
            img = cv2.imread(str(img_path))
            if img is None:
                print(f"Could not load: {img_path}")
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # face_data[i][j] is either a single box (4,) or multiple boxes (N,4)
            boxes = face_data[i][j]
            if boxes.ndim == 1:
                boxes = boxes.reshape(1, -1)  # single face → make it (1, 4)

            face_count = len(boxes)
            for box in boxes:
                x, y, w, h = box[:4].astype(int)
                cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 100), 2)

            axes[i][j].imshow(img)
            axes[i][j].set_title(f"{event} | {face_count} faces", fontsize=9)
            axes[i][j].axis("off")

    plt.tight_layout()
    plt.savefig(str(ROOT / "results" / "sample_annotations.png"), dpi=120)
    plt.show()
    print("Saved → results/sample_annotations.png")

show_samples()

In [ ]:
# Inspect the exact structure of the first few entries
print("event_list[0]:", event_list[0])
print("file_list[0] type:", type(file_list[0]))
print("file_list[0][0]:", file_list[0][0])
print()
print("face_data[0] type:", type(face_data[0]))
print("face_data[0][0] type:", type(face_data[0][0]))
print("face_data[0][0] shape:", np.array(face_data[0][0]).shape)
print("face_data[0][0] value:", face_data[0][0])


In [ ]:
widths, heights = [], []

for i in range(len(event_list)):
    for j in range(len(file_list[i])):
        boxes = face_data[i][j]
        if boxes.ndim == 1:
            boxes = boxes.reshape(1, -1)  # single face
        for box in boxes:
            widths.append(box[2])
            heights.append(box[3])

widths  = np.array(widths)
heights = np.array(heights)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths[widths < 300],  bins=60, color="#7F77DD", edgecolor="none")
axes[0].set_title("Face width distribution")
axes[0].set_xlabel("Width (px)")

axes[1].hist(heights[heights < 300], bins=60, color="#1D9E75", edgecolor="none")
axes[1].set_title("Face height distribution")
axes[1].set_xlabel("Height (px)")

plt.suptitle("WIDER FACE — Face Size Distribution", fontsize=13)
plt.tight_layout()
plt.savefig(str(ROOT / "results" / "face_size_distribution.png"), dpi=120)
plt.show()

print(f"Min face size  : {widths.min():.0f} x {heights.min():.0f} px")
print(f"Max face size  : {widths.max():.0f} x {heights.max():.0f} px")
print(f"Median face    : {np.median(widths):.0f} x {np.median(heights):.0f} px")
print(f"Tiny faces (<20px): {(widths < 20).sum():,} ({(widths<20).mean()*100:.1f}%)")

In [ ]:
# See what we keep at different minimum size thresholds
for min_size in [10, 20, 30, 40, 50]:
    keep = ((widths >= min_size) & (heights >= min_size) & 
            (widths > 0) & (heights > 0))
    print(f"Min {min_size}px → keep {keep.sum():,} faces ({keep.mean()*100:.1f}%)")

In [ ]:
# Face size buckets
buckets = {
    "invalid (<=0px)"  : ((widths <= 0) | (heights <= 0)).sum(),
    "tiny   (1–19px)"  : ((widths.clip(0) >= 1)  & (widths < 20)).sum(),
    "small  (20–49px)" : ((widths >= 20) & (widths < 50)).sum(),
    "medium (50–149px)": ((widths >= 50) & (widths < 150)).sum(),
    "large  (150px+)"  : (widths >= 150).sum(),
}

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#E24B4A", "#EF9F27", "#378ADD", "#1D9E75", "#7F77DD"]
bars = ax.bar(buckets.keys(), buckets.values(), color=colors, edgecolor="none")

for bar, val in zip(bars, buckets.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f"{val:,}", ha="center", fontsize=9)

ax.set_title("Face size distribution by bucket", fontsize=12)
ax.set_ylabel("Number of faces")
plt.tight_layout()
plt.savefig(str(ROOT / "results" / "face_size_buckets.png"), dpi=120)
plt.show()